In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory (Must be /kaggle/working)
OUTPUT_DIR = '/kaggle/working'

# Input files
EVALUATION_RESULTS_FILE = os.path.join(OUTPUT_DIR, 'evaluation_results.csv')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv') # Needed for demand categories

# Output directory for plots
PLOT_DIR = os.path.join(OUTPUT_DIR, 'analysis_plots')
os.makedirs(PLOT_DIR, exist_ok=True)

# Demand Categorization Thresholds (from paper Section 2.1)
ADI_THRESHOLD = 1.32
CV2_THRESHOLD = 0.49

# --- 2. Load Data ---
print("Loading evaluation results and features...")
try:
    eval_df = pd.read_csv(EVALUATION_RESULTS_FILE)
    features_df = pd.read_csv(FEATURES_FILE, usecols=['item_id', 'state_id', 'ADI', 'CV2'])
except FileNotFoundError as e:
    print(f"\nERROR: Could not load required file: {e}. Ensure 'evaluation.py' ran successfully.")
    exit()

# Merge features onto evaluation results for category analysis
results_df = pd.merge(eval_df, features_df, on=['item_id', 'state_id'], how='left')

# --- 3. Overall Performance Comparison (Table 3 Equivalent) ---
print("\n--- Overall Performance (Equivalent to Table 3) ---")

# Calculate average Cost and Service Level for each method
overall_summary = results_df[[
    'MOIC_Cost', 'MOIC_SL',
    'TO_Cost', 'TO_SL',
    'FSH_Cost', 'FSH_SL'
]].mean().reset_index()

overall_summary.columns = ['Metric', 'AverageValue']
overall_summary['Method'] = overall_summary['Metric'].apply(lambda x: x.split('_')[0])
overall_summary['Measure'] = overall_summary['Metric'].apply(lambda x: x.split('_')[1])

# Pivot for better readability
overall_table = overall_summary.pivot(index='Method', columns='Measure', values='AverageValue')
print(overall_table[['Cost', 'SL']]) # Display Cost then SL

# --- 4. Define Demand Categories ---
def categorize_demand(row):
    adi = row['ADI']
    cv2 = row['CV2']
    if adi < ADI_THRESHOLD and cv2 < CV2_THRESHOLD:
        return 'Smooth'
    elif adi >= ADI_THRESHOLD and cv2 < CV2_THRESHOLD:
        return 'Intermittent' # Paper calls this 'Slow', but uses 'Intermittent' in Table 4
    elif adi < ADI_THRESHOLD and cv2 >= CV2_THRESHOLD:
        return 'Erratic'
    else: # adi >= ADI_THRESHOLD and cv2 >= CV2_THRESHOLD
        return 'Lumpy'

results_df['DemandCategory'] = results_df.apply(categorize_demand, axis=1)
print("\nDemand Categories Assigned.")
print(results_df['DemandCategory'].value_counts())

# --- 5. Performance by Demand Category (Table 4 Equivalent) ---
print("\n--- Performance by Demand Category (Equivalent to Table 4) ---")

category_summary = results_df.groupby('DemandCategory')[[
    'MOIC_Cost', 'MOIC_SL',
    'TO_Cost', 'TO_SL',
    'FSH_Cost', 'FSH_SL'
]].mean()

# Display Cost and SL separately or combined
print("\nAverage Cost by Demand Category:")
print(category_summary[['MOIC_Cost', 'TO_Cost', 'FSH_Cost']])

print("\nAverage Service Level (SL) by Demand Category:")
print(category_summary[['MOIC_SL', 'TO_SL', 'FSH_SL']])

# --- 6. Performance by Region (State) ---
print("\n--- Performance by State ---")
state_summary = results_df.groupby('state_id')[[
    'MOIC_Cost', 'MOIC_SL',
    'TO_Cost', 'TO_SL',
    'FSH_Cost', 'FSH_SL'
]].mean()

print("\nAverage Cost by State:")
print(state_summary[['MOIC_Cost', 'TO_Cost', 'FSH_Cost']])

print("\nAverage Service Level (SL) by State:")
print(state_summary[['MOIC_SL', 'TO_SL', 'FSH_SL']])


# --- 7. Visualizations (Figures 7 & 8 Equivalents) ---
print("\nGenerating comparison plots...")
sns.set_style("whitegrid")

# Figure 7a: Distribution of Total Cost
plt.figure(figsize=(10, 6))
cost_data = results_df[['MOIC_Cost', 'TO_Cost', 'FSH_Cost']]
# Handle potential extreme outliers for better visualization
cost_data = cost_data[cost_data < cost_data.quantile(0.99)] # Remove top 1% outliers
sns.boxplot(data=cost_data, palette='viridis')
plt.title('Distribution of Realized Total Inventory Cost per Series (Outliers Removed)')
plt.ylabel('Total Cost ($)')
plt.xlabel('Method')
plt.xticks(ticks=[0, 1, 2], labels=['MOIC', 'TO', 'FSH'])
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'figure7a_cost_distribution.png'))
print(f"  Saved Cost Distribution plot to {PLOT_DIR}")
plt.close()

# Figure 7b: Distribution of Service Level
plt.figure(figsize=(10, 6))
sl_data = results_df[['MOIC_SL', 'TO_SL', 'FSH_SL']]
sns.boxplot(data=sl_data, palette='viridis')
plt.title('Distribution of Realized Service Level per Series')
plt.ylabel('Service Level')
plt.xlabel('Method')
plt.ylim(0.8, 1.01) # Set Y-axis limits typical for SL
plt.xticks(ticks=[0, 1, 2], labels=['MOIC', 'TO', 'FSH'])
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'figure7b_sl_distribution.png'))
print(f"  Saved Service Level Distribution plot to {PLOT_DIR}")
plt.close()

# Figure 8 (Example): Cost Distribution by Demand Category
plt.figure(figsize=(14, 8))
cost_data_melt = pd.melt(results_df,
                         id_vars=['DemandCategory'],
                         value_vars=['MOIC_Cost', 'TO_Cost', 'FSH_Cost'],
                         var_name='Method', value_name='Cost')
cost_data_melt['Method'] = cost_data_melt['Method'].str.replace('_Cost', '')
# Remove outliers for plotting
cost_data_melt = cost_data_melt[cost_data_melt['Cost'] < cost_data_melt['Cost'].quantile(0.99)]

sns.boxplot(x='DemandCategory', y='Cost', hue='Method', data=cost_data_melt, palette='viridis')
plt.title('Distribution of Realized Total Cost by Demand Category (Outliers Removed)')
plt.ylabel('Total Cost ($)')
plt.xlabel('Demand Category')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'figure8_cost_by_category.png'))
print(f"  Saved Cost Distribution by Category plot to {PLOT_DIR}")
plt.close()

print("\nAnalysis complete. Plots saved in:", PLOT_DIR)

# --- 8. Conclusion Summary (Printed Output) ---
print("\n--- Summary ---")
print("Analysis complete. Compare the printed tables and generated plots to the paper's findings.")
print("Key metrics (Average Cost and SL) are available in the printed tables.")
print("Check the '/kaggle/working/analysis_plots' directory for visualizations.")


Loading evaluation results and features...

ERROR: Could not load required file: [Errno 2] No such file or directory: '/kaggle/working/evaluation_results.csv'. Ensure 'evaluation.py' ran successfully.


NameError: name 'eval_df' is not defined

: 

: 

: 

: 

: 